# Unsupervised Learning in R - Part B (Submission File)

# Enter in your Name:

## About the Dataset
This is a small dataset from 1973 on protein consumption from nine different food groups in 25 countries in Europe. The goal is to group the countries based on patterns in their protein consumption.

References: 1. Zumel, N. and Mount, J. “Practical Data Science with R”, Manning Publications, 2014.


## Original Link
https://github.com/is6481/lab5.git

In [ ]:
install.packages("fpc")
install.packages("RMySQL")

In [ ]:

library(ggplot2)
library(fpc)
library(tidyverse)
library(ggrepel)
library(RMySQL)
library(magrittr)

mm <- src_mysql(
  host='[updateme]',
  port=3306,
  user='admin',
  password='[updateme]',
  dbname='unsupervised_learning'
)

In [ ]:
#here is the table object (not the data yet)
print(mm)

In [ ]:
#now we read in the table called protein into a data object
d_protein <- tbl(mm,'protein') %>% collect() # Get data from the database

In [ ]:
#view what the data looks like

head(d_protein)

In [ ]:
# This code block performs the following steps:

# 1. Transforms the 'd_protein' dataframe from wide to long format using pivot_longer.
#    - It selects all columns except 'Country' and pivots them into two columns: 'type' and 'value'.
#    - 'type' will contain the original column names (e.g., 'RedMeat', 'WhiteMeat', etc.).
#    - 'value' will contain the corresponding values for each type.

density_data <- d_protein %>% 
    pivot_longer(cols=-Country, names_to='type')
head(density_data)
# 2. Creates a density plot using ggplot2.
#    - The x-axis represents the 'value' column from the long-format dataframe.
#    - geom_density() adds a density plot with a specified fill color and transparency.
#    - facet_wrap() creates separate plots for each 'type' with free scales.
#    - labs() sets the labels for the x and y axes.
#    - theme_minimal() applies a minimal theme to the plot.
#    - theme() customizes the theme by removing minor grid lines on the y-axis.

ggplot(density_data, aes(x=value)) + 
    geom_density(fill='#F6FEAA', alpha=0.6) + 
    facet_wrap(~type, scales='free') +
    labs(x='Value', y='Density Function') +
    theme_minimal() +
    theme(panel.grid.minor.y=element_blank())

## A. Understanding Density Plots

Answer the following questions:

A1. Look at the density plot for white meat, what does it mean? In other words, what does it tell you about white meat consumption?

A2. What would a bad set of density plots look like for a cluster solution?

A3. Do you think this set of variables will provide an interesting solution? Why?


## B. Variable Choice and Transformation

We could narrow down our set of variables, for instance we could create a single variable for meat by combining the two meat categories (e.g. using a simple sum, or taking the row-wise maximum between RedMeat and WhiteMeat).

B1. What do you think it would do to the solution to collapse the two meat variables?


## C. Cluster Algorithms

As discussed in the lab, and presented on Datacamp, there are several different versions of clustering algorithms available to the analyst. They all work differently. The following code runs the k-means cluster algorithm as seen in the lab writeup. 

In [ ]:
#first we scale the variables to center them

var_list <- c("RedMeat", "WhiteMeat", "Eggs", "Milk", "Fish", "Cereals", "Starch", "Nuts", "FrAndVeg") 

m_protein <- as.data.frame(scale(d_protein[, var_list])) # filters to only the attributes we need and scales them
names(m_protein) <- paste0('st_',var_list)

all_protein <- cbind(d_protein,m_protein)

In [ ]:
set.seed(42)
cluster_model <- kmeans(m_protein, centers=5)
cluster_model$centers %>% t() %>% knitr::kable(caption='k-means Cluster Solution')
print(table(cluster_model$cluster))

C1. Give a descriptive name to each of the clusters.

C2. Which countries fall in which clusters? See if you can figure out the R-Code to answer that question. {Hint, you have to combine the original dataset with the clusters generated to do this}


## Hierarchical Clustering Analysis

Following is a cluster analysis of the same data but using a different clustering algorithm. Look at the profiling table and answer the questions below.

In [ ]:
d <- dist(m_protein,method='euclidean')
hc1 <- hclust(d,method='ward.D')
h_profiling <- m_protein
h_profiling$hclust <- cutree(hc1,k=5)

h_profile_table <- h_profiling %>% pivot_longer(cols=-hclust) %>%
  pivot_wider(names_from=name,values_from=value,values_fn=list(value=mean))

h_profile_table %>% knitr::kable(caption='Hierarchical Cluster Solution')

C2. Give a descriptive name to each of the clusters.


C3. Describe the differences between the two clustering solutions (Hierarchical vs K-Means).


In [1]:
#enter in the new code to view the cluster name and the country name as a table in here

## D. Change the number of clusters

Copy either of the clustering code segments from above, change the number of clusters and do the following.

D1. Name each of the clusters

D2. Describe the differences between the two solutions. Provide a comparison or matching e.g. mention which clusters appear in both analyses.

D3. Mention which number of clusters you prefer and why.

In [ ]:
#enter in the copied code from here


## To Turn In

When you are finished, please click the "Report" button above to preview the final document. *Make sure your responses to the questions are shown*. When finished, click on [File>Export to PDF>Report View] and upload to your work to Canvas.